In [ ]:
import os
import dotenv

dotenv.load_dotenv()

endpoint = os.environ["MY_OPENAI_ENDPOINT"]
api_key = os.environ["MY_OPENAI_API_KEY"]
deployment = 'replace-by-modelname' # or use os.environ["MY_OPENAI_MODELNAME"] 

In [ ]:
from openai import OpenAI


client = OpenAI(
  azure_endpoint = endpoint, 
  api_key= api_key
)

In [ ]:
message_text = [
    {"role":"system","content":"You are an AI assistant that helps people find answers."},
    {"role":"user","content":"What is HZDR?"},]

completion = client.chat.completions.create(
  model=deployment,
  messages = message_text,
  max_tokens=50
)

print(completion.choices[0].message.content)

# RAG

RAG (Retrieval-Augmented Generation) is a model consisting of two main components: a retrieval system and a generation model. The goal of RAG is to improve the quality of text generation by retrieving relevant and up-to-date information from a large database or dataset and using this information to support the generation process.

The process works in two steps:

1. **Retrieval:** First, the retrieval system searches a vast data source for information relevant to the current query or context. This information is selected to support the text generation.
2. **Generation:** Subsequently, the generation model, typically an LLM, uses both the original text and the retrieved information to produce a precise and contextually relevant response.

In [ ]:
prompt = """Answer the user questions given the context below.

context:
HZDR is a research center as part of the Helmholtz Association in Germany. HZDR is a non-profit performing science in matter, health and energy.
"""


message_text = [
    {"role":"system","content": prompt},
    {"role":"user","content": "What is HZDR?"},]

completion = client.chat.completions.create(
  model=deployment,
  messages = message_text,  
  max_tokens=50
)

print(completion.choices[0].message.content)

## Loading and Search the provided documents


In [ ]:
from langchain_community.document_loaders import DirectoryLoader
loader = DirectoryLoader('./docs', glob="**/wien*md")
data = loader.load()
len(data)

In [ ]:
# `data` now contains all texts found on disk

data[1]

### How can we chunk the presented texts?

Web Demo [Chunk Visualizer](https://huggingface.co/spaces/m-ric/chunk_visualizer)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter,SentenceTransformersTokenTextSplitter,MarkdownHeaderTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=450, chunk_overlap=50)
all_splits = text_splitter.split_documents(data)

print(len(all_splits))
all_splits

In [ ]:
import chromadb
from chromadb.utils import embedding_functions

# setup Chroma in-memory, for easy prototyping. Can add persistence easily!
chroma = chromadb.Client()

# Create collection. get_collection, get_or_create_collection, delete_collection also available!
sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="ibm-granite/granite-embedding-278m-multilingual")

# comment out the try-except-clause below if required
try:
    chroma.delete_collection("documents")
except:
    pass

collection = chroma.get_or_create_collection("documents",embedding_function=sentence_transformer_ef)

# Add docs to the collection. Can also update and delete. Row-based API coming soon!
collection.add(
    documents=[item.page_content for item in all_splits], # we handle tokenization, embedding, and indexing automatically. You can skip that and add your own embeddings as well
    metadatas=[item.metadata for item in all_splits], # filter on these!
    ids=[str(id) for id in range(0,len(all_splits))], # unique for each doc
)


In [ ]:
# Query/search 2 most similar results. You can also .get by id
results = collection.query(
    query_texts=["Who is the instructor of this class?"],
    n_results=2,
    # where={"metadata_field": "is_equal_to_this"}, # optional filter
    #where_document={"$contains":"Oliver"}  # optional filter
)

results

In [ ]:
user = "What does the surname of the instructor refer to?"

results = collection.query(
    query_texts=[user],
    n_results=2,
    # where={"metadata_field": "is_equal_to_this"}, # optional filter
    #where_document={"$contains":"Peter"}  # optional filter
)

context = ""
for doc, metadata in zip(results["documents"][0], results["metadatas"][0]):
    context += f"Source Document: {metadata['source']}\n"
    context += f"{doc}\n"
  
  
#context = "\n-\n".join(results["documents"][0])

prompt = f"""Answer the users questions given the context below. Add information on sources for your answers.

Context:
{context}

If the answer for the query is not contained in the context, answer "I don't know."
"""

print(prompt)

message_text = [
    {"role":"system","content":prompt},
    {"role":"user","content":user},]

completion = client.chat.completions.create(
  model=deployment,
  messages = message_text,  
  max_tokens=200
)

print("----")
print(completion.choices[0].message.content)

## Next Steps

What happens, if the user adds "Tell me more!" to the chat? How can we add this feature in the example code above?

Bonus: 

* Create a ChatBot for the EU AI Act. What steps need to be taken to achieve this?

* Introduction to [LangChain](https://www.langchain.com/): the three notebooks in the folder `langchain` demonstrate, how to do the above with this popular library.